# Step 1: Data Ingestion And Quality Checks

This notebook establishes the project baseline for the Instacart market basket analysis.

It covers:
- loading the three raw CSV files
- parsing timestamps and creating time features
- auditing missing values, duplicates, and row counts
- saving a lightweight data audit for downstream notebooks


In [1]:
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")


In [2]:
from pathlib import Path


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, current.parent):
        if (candidate / "orders.csv").exists() and (candidate / "products.csv").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root that contains the Instacart CSV files.")


PROJECT_ROOT = find_project_root()
OUTPUT_DIR = PROJECT_ROOT / "outputs"
ANALYSIS_DIR = OUTPUT_DIR / "analysis"
FIGURE_DIR = OUTPUT_DIR / "figures"
QUALITY_DIR = OUTPUT_DIR / "quality"

for path in (OUTPUT_DIR, ANALYSIS_DIR, FIGURE_DIR, QUALITY_DIR):
    path.mkdir(parents=True, exist_ok=True)


In [3]:
orders_path = PROJECT_ROOT / "orders.csv"
products_path = PROJECT_ROOT / "products.csv"
order_products_path = PROJECT_ROOT / "all_order_products.csv"

orders = pd.read_csv(orders_path, parse_dates=["timestamp"])
products = pd.read_csv(products_path)

orders["order_date"] = orders["timestamp"].dt.date
orders["order_hour"] = orders["timestamp"].dt.hour
orders["order_day_name"] = orders["timestamp"].dt.day_name()
orders["order_day_of_week"] = orders["timestamp"].dt.dayofweek
orders["order_month"] = orders["timestamp"].dt.to_period("M").astype(str)

orders.head()


,order_id,user_id,timestamp,order_date,order_hour,order_day_name,order_day_of_week,order_month
0,2539329,1,2018-01-03 08:00:00,2018-01-03,8,Wednesday,2,2018-01
1,2398795,1,2018-01-18 07:00:00,2018-01-18,7,Thursday,3,2018-01
2,473747,1,2018-02-08 12:00:00,2018-02-08,12,Thursday,3,2018-02
3,2254736,1,2018-03-09 07:00:00,2018-03-09,7,Friday,4,2018-03
4,431534,1,2018-04-06 15:00:00,2018-04-06,15,Friday,4,2018-04


In [4]:
sample_order_products = pd.read_csv(order_products_path, nrows=100_000)

row_counts = {
    "orders": len(orders),
    "products": len(products),
    "order_products_sample": len(sample_order_products),
}

missing_summary = pd.DataFrame(
    {
        "orders_missing": orders.isna().sum(),
        "products_missing": products.isna().sum(),
    }
).fillna(0).astype(int)

duplicate_summary = pd.DataFrame(
    [
        {"table": "orders", "duplicate_rows": int(orders.duplicated().sum())},
        {"table": "products", "duplicate_rows": int(products.duplicated().sum())},
        {"table": "order_products_sample", "duplicate_rows": int(sample_order_products.duplicated().sum())},
    ]
)

audit_summary = pd.DataFrame(
    [
        {"metric": "orders_row_count", "value": len(orders)},
        {"metric": "products_row_count", "value": len(products)},
        {"metric": "unique_users", "value": orders["user_id"].nunique()},
        {"metric": "unique_orders", "value": orders["order_id"].nunique()},
        {"metric": "unique_products", "value": products["product_id"].nunique()},
        {"metric": "min_timestamp", "value": str(orders["timestamp"].min())},
        {"metric": "max_timestamp", "value": str(orders["timestamp"].max())},
    ]
)

print(row_counts)
missing_summary


{'orders': 3346083, 'products': 49688, 'order_products_sample': 100000}


,orders_missing,products_missing
aisle_id,0,0
department_id,0,0
order_date,0,0
order_day_name,0,0
order_day_of_week,0,0
order_hour,0,0
order_id,0,0
order_month,0,0
product_id,0,0
product_name,0,0


In [5]:
duplicate_summary


,table,duplicate_rows
0,orders,0
1,products,0
2,order_products_sample,0


In [6]:
audit_summary.to_csv(QUALITY_DIR / "data_audit_summary.csv", index=False)
missing_summary.to_csv(QUALITY_DIR / "missing_value_summary.csv")
duplicate_summary.to_csv(QUALITY_DIR / "duplicate_summary.csv", index=False)

audit_summary


,metric,value
0,orders_row_count,3346083
1,products_row_count,49688
2,unique_users,206209
3,unique_orders,3346083
4,unique_products,49688
5,min_timestamp,2018-01-01 00:00:00
6,max_timestamp,2019-01-07 23:00:00


## Step 1 Outcome

After running this notebook you will have:
- validated the basic schema of each file
- generated reusable time-based features from `orders.csv`
- exported quality summaries into `outputs/quality/`
